# Labeling the Dataset using the trained student

What we do:
1) Run the student model on the dataset of inputs
2) Analyse the student dataset and label it


Student: Qwen2.5-1.5B-Instruct

In [12]:
import dotenv

dotenv.load_dotenv()

True

In [13]:
from core.types import *
from core.utils.huggingface_client import HuggingFaceClient
from core.utils.huggingface_inference_client import HuggingFaceInferenceClient
from core.utils.ollama_inference_client import OllamaInferenceClient
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.utils.doom_game_state import DoomGameState, MonsterType, WeaponName, AimedAtType
from sklearn.cluster import DBSCAN
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Iterable
from pathlib import Path
from ollama import ChatResponse
from openai.types.responses import Response as OpenAIResponse

import os
import json
import numpy as np
import pandas as pd

In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
inputs = LLMCommandingInput.load_inputs(
    path=Path("data/inputs/inputs.json"),
    gstype=DoomGameState
)

inputs_lookup = {inp.id: inp for inp in inputs}

In [16]:
# For now, only extract inputs to label
selected_inputs = [
    inp
    for inp in inputs
    if inp.selected_for_labelling
]

print(f"Selected inputs: {len(selected_inputs)}/{len(inputs)}")

Selected inputs: 50/2872


In [17]:
student_client = HuggingFaceInferenceClient[LLMCommandingInput, LLMCommandingOutput](
    #model = "Qwen/Qwen2.5-1.5B-Instruct",
    model = "./data/huggingface/training/final",
    max_output_tokens=800,
    temperature=0.0,
    working_dir=Path('data/huggingface'),
    use_flash_attention_2=False,
    load_in_4bit=True
)

🚀 Initialized HuggingFaceInferenceClient for ./data/huggingface/training/final
   Device: cuda
   Flash Attention 2: False
   Quantization: 4-bit


In [18]:
# Prepare Prompt (same as training)
system_prompt = "You are a game command parser that converts natural language commands into DSL instructions."

In [19]:
def format_input(inp: LLMCommandingInput) -> str:
    game_state = inp.game_state.state.to_prompt_ready()
    command = inp.user_command.command.command
    return f"Game State:\n{game_state}\nCommand:\n{command}"


def parse_output(response: str, input_id: str, latency: float) -> LLMCommandingOutput:
    return LLMCommandingOutput(
        input_id=input_id,
        actions=response,
        reason=None,
        latency=latency,
    )


def get_id(gse: LLMCommandingInput, idx: int) -> str:
    return gse.id

In [9]:
print(system_prompt)
print(format_input(inputs[3]))

You are a game command parser that converts natural language commands into DSL instructions.
Game State:
AIMED_AT:
  type: Wall
  distance: 330.86
  interactable: yes

MONSTERS (count=0):

INVENTORY:
  current_slot: 2
  weapons:
    - (1, Fist, 0)
    - (2, Pistol, 50)
Command:
Go press that switch ahead


In [20]:
outputs = student_client.process(
    dataset=selected_inputs, # TO CHANGE
    system_prompt=system_prompt,
    tools = [], # No tools at level 3
    format_input=format_input,
    parse_output=parse_output,
    get_id=get_id,
    # batch_size=200,
)


📦 Loading model: ./data/huggingface/training/final


The tokenizer you are loading from './data/huggingface/training/final' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


   📉 Using 4-bit quantization (NF4)
   ✓ Model loaded successfully!

🔄 Processing 50 items sequentially


Processing items:   0%|          | 0/50 [00:00<?, ?it/s]

  [1] Message building: 0.000s
  [2] Chat template: 0.001s
  [2a] Prompt length: 369 chars
  [3] Tokenization: 0.000s
  [3a] Input tokens: 98


Processing items:   2%|▏         | 1/50 [00:29<23:52, 29.23s/it]

  [4] Generation: 29.226s
  [4a] Output tokens: 898
  [5] Decoding: 0.000s
  [5a] Response length: 2162 chars
  [TOTAL]: 29.227s

  [1] Message building: 0.000s
  [2] Chat template: 0.000s
  [2a] Prompt length: 408 chars
  [3] Tokenization: 0.001s
  [3a] Input tokens: 118


Processing items:   2%|▏         | 1/50 [00:37<30:13, 37.00s/it]


KeyboardInterrupt: 